# 02 — Sample Design

In [ ]:
# Cell 1 — setup
import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parent / "src"))
from scorelab import config, data
con = data.connect()            # read-only DuckDB connection to data/lc.duckdb

In [ ]:
# Cell 2 — sanity check
print(con.sql("SHOW TABLES"))
print(con.sql("SELECT COUNT(*) AS n, MIN(issue_d), MAX(issue_d) FROM accepted"))

## What this notebook settles

Five decisions, each with the evidence above it and a cell for your reasoning below it. **Write the reasoning before you run the evidence cells**, then compare. Everything downstream (NB03 onward) reads the label from `v_target`, so once these are settled, run `python scripts/refresh_views.py` if you change `config.py`, and record the decision in `docs/decisions.md`.

| # | Decision | Where it lives |
|---|---|---|
| 1 | Performance window (24 months) | `config.PERFORMANCE_WINDOW_MONTHS`, `config.PERFORMANCE_SNAPSHOT` |
| 2 | Bad event and indeterminates | `config.BAD_STATUSES`, `config.DELINQUENT_STATUSES`, timing constants |
| 3 | Exclusions | `exclusion` column in `v_target` (credit-policy rows today) |
| 4 | Split and the censored monitor range | `config.TRAIN_VINTAGES` … `MONITOR_VINTAGES`, `EARLY_WINDOW_MONTHS` |
| 5 | Sampling | full data vs. oversampled bads, prior correction |

Lending Club gives one snapshot per loan, not a payment history, so the month a loan reached 90+ DPD is reconstructed from `last_pymnt_d` (first miss = last payment + 1 month, 90 DPD = first miss + 3). That reconstruction is the proxy-target caveat the pilot report must state against VantageScore's observed 90+ DPD definition.

In [ ]:
# The definition currently in force (edit config.py, then scripts/refresh_views.py, to change it)
from scorelab import target
import pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)

print("snapshot          ", config.PERFORMANCE_SNAPSHOT)
print("window (months)   ", config.PERFORMANCE_WINDOW_MONTHS)
print("bad statuses      ", config.BAD_STATUSES)
print("delinquent (open) ", config.DELINQUENT_STATUSES)
print("timing            last payment +%d = first miss; +%d = 60 DPD; +%d = 90 DPD" % (
    config.MONTHS_LAST_PAYMENT_TO_FIRST_MISS, config.MONTHS_FIRST_MISS_TO_60DPD, config.MONTHS_FIRST_MISS_TO_90DPD))
print("early indicator   60+ DPD within", config.EARLY_WINDOW_MONTHS, "months")
print("incumbent         ", config.INCUMBENT_SCORE_NAME, "=", config.INCUMBENT_SCORE_SQL)

cols = [r[0] for r in con.execute("DESCRIBE v_target").fetchall()]
assert "label" in cols, "v_target is the old stub: close this kernel and run `python scripts/refresh_views.py`"
print("\nv_target columns:", cols)

### Decision 1 — Performance window

Questions to answer in the cell below, before running the evidence: How much of the eventual charge-off is visible by month 24? Where do the vintage curves flatten? What does a shorter window buy (more observable vintages) and cost (early bads only, lower bad rate, less like a bureau score)? Which vintages does the 24-month window leave censored?

**Your reasoning:**

_(write here)_

In [ ]:
# Evidence 1a — vintage curves: cumulative share reaching 90+ DPD by month since issue
vc = target.vintage_curves(con, max_month=36)
table = vc[vc.horizon.isin([6, 12, 18, 24, 30, 36])].pivot(index="vintage_year", columns="horizon", values="cum_bad_rate")
display(table)

fig, ax = plt.subplots(figsize=(9, 5))
for vy, g in vc.groupby("vintage_year"):
    if vy >= min(config.TRAIN_VINTAGES):
        ax.plot(g.horizon, g.cum_bad_rate, label=str(vy))
ax.axvline(config.PERFORMANCE_WINDOW_MONTHS, color="grey", ls="--", lw=1)
ax.set_xlabel("months since issue"); ax.set_ylabel("cumulative 90+ DPD rate"); ax.set_title("Vintage curves (observable loans at each horizon)")
ax.legend(title="vintage"); plt.show()

In [ ]:
# Evidence 1b — what alternative windows would do, on the modelling vintages
display(target.window_sensitivity(con, windows=(12, 18, 24, 30, 36)))

### Decision 2 — Bad event and indeterminates

The rule in force: bad = 90+ DPD reached inside the window (Charged Off / Default, or an open delinquency, timed from the last payment); indeterminate = first miss inside the window but 90+ only after it; good = everything else observable, including loans that defaulted after the window closed.

Questions: Is a loan that charged off in month 30 of a 60-month term "good" for a 24-month score? (Under VantageScore's definition, yes.) Is the indeterminate share small enough to exclude without bias, and what does the sensitivity in NB08 need to show? Should `Late (31-120 days)` at the snapshot ever count as bad for an observable loan?

**Your reasoning:**

_(write here)_

In [ ]:
# Evidence 2a — label counts and rates by vintage, split and term
display(target.label_summary(con, by="vintage_year"))
display(target.label_summary(con, by="split"))
display(target.label_summary(con, by="term_months"))

In [ ]:
# Evidence 2b — who the indeterminates are
display(target.indeterminate_profile(con))

### Decision 3 — Exclusions

Today only the pre-2010 "Does not meet the credit policy" rows are flagged. Candidates to consider: joint applications (`application_type`), 60-month loans (a different product, or a segment?), loans with a null `last_pymnt_d` that are not charged off, states or purposes with too few loans to level.

**Your reasoning:**

_(write here)_

In [ ]:
# Evidence 3 — exclusion flags and candidate exclusions by split
display(con.sql("""
    SELECT split, exclusion, COUNT(*) AS n FROM v_split GROUP BY 1, 2 ORDER BY 1, 2
""").df())
display(con.sql("""
    SELECT s.split, a.application_type, s.term_months, COUNT(*) AS n,
           ROUND(AVG(CASE WHEN s.label = 'bad' THEN 1 WHEN s.label = 'good' THEN 0 END), 4) AS bad_rate
    FROM v_split s JOIN v_accepted_asof a USING (id)
    WHERE s.observable = 1 AND s.split IN ('train', 'validate', 'oot')
    GROUP BY 1, 2, 3 ORDER BY 1, 2, 3
""").df())

### Decision 4 — Split and the censored monitor range

Train 2012–2014, validate 2015, out-of-time 2016, monitor 2017Q1–2018Q4. At the 2019-03 snapshot only 2017Q1 is observable at 24 months; the rest of the monitor range carries the 12-month early indicator (60+ DPD) and the distribution metrics (PSI/CSI) instead. Is 2016 the right out-of-time year, given its bad rate is the highest in the data? Should validate be used for early stopping and calibration only, never for model selection against OOT?

**Your reasoning:**

_(write here)_

In [ ]:
# Evidence 4 — split sizes, observability and the early indicator by quarter
display(con.sql("""
    SELECT split, COUNT(*) AS n, SUM(observable) AS n_observable_24m, SUM(early_observable) AS n_observable_12m,
           ROUND(AVG(CASE WHEN label = 'bad' THEN 1 WHEN label = 'good' THEN 0 END), 4) AS bad_rate_24m,
           ROUND(AVG(early_bad), 4) AS early_bad_rate_12m
    FROM v_split WHERE exclusion IS NULL GROUP BY 1 ORDER BY 1
""").df())
display(target.label_summary(con, by="vintage_quarter").query("vintage_quarter >= '2016Q1'"))

### Decision 5 — Sampling

With roughly 40k bads in train there is no need to oversample; the full data fits on an ordinary machine. If you do subsample goods for speed, the intercept must be corrected afterwards (`target.prior_correct_intercept`), or every odds-to-score table is off by the sampling ratio.

**Your reasoning:**

_(write here)_

In [ ]:
# Evidence 5 — bads per split and the size of the full modelling frame; prior-correction example
display(con.sql("""
    SELECT split, SUM(CASE WHEN label = 'bad' THEN 1 ELSE 0 END) AS bads,
           SUM(CASE WHEN label = 'good' THEN 1 ELSE 0 END) AS goods
    FROM v_split WHERE split IN ('train', 'validate', 'oot') AND exclusion IS NULL GROUP BY 1 ORDER BY 1
""").df())
import math
print("Fit on a 1:1 sample when the population bad rate is 10%  ->  intercept shift of",
      round(target.prior_correct_intercept(0.0, 0.5, 0.10), 3), "(= -ln 9)")
# A 10% exploration sample of the modelling frame:
frame = target.modeling_frame(con, sample_frac=0.10)
print(frame.shape); display(frame.groupby("split")["bad"].agg(["mean", "count"]))

## Record

Paste the decisions into `docs/decisions.md` (dated) and the definition into `docs/model_card.md`. The cell below prints a draft with the numbers filled in; edit the wording, not the numbers.

In [ ]:
s = target.label_summary(con, by="split").set_index("split")
ws = target.window_sensitivity(con, windows=(config.PERFORMANCE_WINDOW_MONTHS,)).iloc[0]
print(f"""## {pd.Timestamp.today():%Y-%m-%d} — Target definition and sample design (NB02)

- Performance observed at the {config.PERFORMANCE_SNAPSHOT} snapshot; window {config.PERFORMANCE_WINDOW_MONTHS} months.
- Bad = 90+ DPD inside the window, reconstructed from last_pymnt_d (+{config.MONTHS_LAST_PAYMENT_TO_FIRST_MISS} first miss,
  +{config.MONTHS_FIRST_MISS_TO_90DPD} to 90 DPD); statuses {config.BAD_STATUSES} and open delinquencies {config.DELINQUENT_STATUSES}.
- Indeterminate = first miss inside the window, 90+ after it: excluded from fitting, reported in NB08 sensitivity.
- Loans delinquent only after the window are good (vendor-style definition). Charge-offs captured by the window:
  {ws.share_of_eventual_chargeoffs_captured:.1%} of those observed at the snapshot.
- Bad rate excluding indeterminates: train {s.loc['train','bad_rate_excl_indet']:.2%}, validate {s.loc['validate','bad_rate_excl_indet']:.2%},
  oot {s.loc['oot','bad_rate_excl_indet']:.2%}; indeterminate share about {ws.indeterminate_share:.1%}.
- Split: train {config.TRAIN_VINTAGES}, validate {config.VALIDATE_VINTAGE}, oot {config.OOT_VINTAGE}, monitor {config.MONITOR_VINTAGES}
  (censored at 24 months; monitored with 60+ DPD within {config.EARLY_WINDOW_MONTHS} months and PSI/CSI).
- Incumbent score for the pilot: {config.INCUMBENT_SCORE_NAME} = {config.INCUMBENT_SCORE_SQL}.
- Exclusions: credit-policy rows; <add yours>.
- Sampling: <full data / sample + prior correction>.
""")